# Calibrate AA-CLIP decision thresholds

The official AA-CLIP paper and repository report threshold-independent AUROC/AP metrics and do not publish a binary anomaly decision threshold. This notebook mirrors the AnomalyCLIP threshold workflow and creates a **custom, explicitly labeled fallback** for flip rate, targeted success, FPR, FNR, and qualitative selection. For each dataset and category, it takes the 95th percentile of AA-CLIP's official aggregated image anomaly score on clean normal training images only. Test images, test labels, anomalies, and adversarial images are never used.

The zero-shot mapping is the same as the evaluator: MVTec uses `TrainOnVisA`, and VisA uses `TrainOnMVTec` from [parsagh1383/aa-clip-checkpoints-main](https://www.kaggle.com/datasets/parsagh1383/aa-clip-checkpoints-main).

In [ ]:
import hashlib
import shutil
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE REPOSITORIES AND INSTALL DEPENDENCIES =====')
WORKING = Path('/kaggle/working')
EXPERIMENT_ROOT = WORKING / 'adversarial-robustness'
AACLIP_ROOT = WORKING / 'AA-CLIP'
EXPERIMENT_REPO_URL = 'https://github.com/Parsagh05/adversarial-robustness.git'
AACLIP_REPO_URL = 'https://github.com/Mwxinnn/AA-CLIP.git'
AACLIP_COMMIT = '53db195f230442aa118c246876c94ba1c76139cc'

def clone_or_update(url, destination, commit=None):
    if destination.exists():
        subprocess.run(['git', '-C', str(destination), 'fetch', '--all', '--tags'], check=True)
    else:
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    if commit:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    else:
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)

clone_or_update(EXPERIMENT_REPO_URL, EXPERIMENT_ROOT)
clone_or_update(AACLIP_REPO_URL, AACLIP_ROOT, AACLIP_COMMIT)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'requirements.txt')
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'einops==0.7.0', 'ftfy==6.2.0', 'ipdb>=0.13', 'kornia==0.6.9',
    'tiktoken==0.7.0', 'timm==0.6.12'
], check=True)
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))

BASE_MODEL_NAME = 'ViT-L-14-336px.pt'
BASE_MODEL_SHA256 = '3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02'
BASE_MODEL_URL = (
    'https://openaipublic.azureedge.net/clip/models/'
    + BASE_MODEL_SHA256 + '/' + BASE_MODEL_NAME
)
BASE_MODEL_PATH = AACLIP_ROOT / 'model' / BASE_MODEL_NAME

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

attached_base = next(
    (path for path in Path('/kaggle/input').rglob(BASE_MODEL_NAME) if path.is_file()),
    None,
)
if not BASE_MODEL_PATH.is_file() or sha256(BASE_MODEL_PATH) != BASE_MODEL_SHA256:
    if attached_base is not None:
        if sha256(attached_base) != BASE_MODEL_SHA256:
            raise RuntimeError(f'Attached base model has the wrong SHA256: {attached_base}')
        shutil.copy2(attached_base, BASE_MODEL_PATH)
    else:
        import torch
        torch.hub.download_url_to_file(
            BASE_MODEL_URL, str(BASE_MODEL_PATH), hash_prefix=BASE_MODEL_SHA256, progress=True
        )
if sha256(BASE_MODEL_PATH) != BASE_MODEL_SHA256:
    raise RuntimeError(f'Base-model checksum mismatch: {BASE_MODEL_PATH}')
print('Experiment code:', EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline')
print('Official AA-CLIP:', AACLIP_ROOT)
print('Verified base model:', BASE_MODEL_PATH)

In [ ]:
import torch

print('===== STEP 2: RESOLVE DATASETS AND ZERO-SHOT AA-CLIP CHECKPOINTS =====')
CHECKPOINT_DATASET_URL = 'https://www.kaggle.com/datasets/parsagh1383/aa-clip-checkpoints-main'

def first_existing_directory(paths, label):
    for path in paths:
        if path.is_dir():
            return path
    raise FileNotFoundError(f'{label} was not found. Checked: {paths}')

MVTEC_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection'),
    Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection'),
], 'MVTec AD')
VISA_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922'),
    Path('/kaggle/input/visa-ad/VisA_20220922'),
], 'VisA')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')

checkpoint_roots = [
    path for path in (
        Path('/kaggle/input/aa-clip-checkpoints-main'),
        Path('/kaggle/input/datasets/parsagh1383/aa-clip-checkpoints-main'),
    ) if path.is_dir()
]
if not checkpoint_roots:
    raise FileNotFoundError(
        'Attach parsagh1383/aa-clip-checkpoints-main as a Kaggle input: ' + CHECKPOINT_DATASET_URL
    )

def resolve_training_checkpoint(training_name):
    matches = []
    for root in checkpoint_roots:
        for image_path in root.rglob('image_adapter.pth'):
            if training_name.lower() in {part.lower() for part in image_path.parts}:
                matches.append(image_path.parent)
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {training_name} checkpoint directory, found: {matches}')
    directory = matches[0]
    image_path = directory / 'image_adapter.pth'
    text_path = directory / 'text_adapter.pth'
    return image_path, text_path if text_path.is_file() else None

TRAIN_ON_MVTEC_IMAGE, TRAIN_ON_MVTEC_TEXT = resolve_training_checkpoint('TrainOnMVTec')
TRAIN_ON_VISA_IMAGE, TRAIN_ON_VISA_TEXT = resolve_training_checkpoint('TrainOnVisA')
MODEL_KWARGS_BY_TARGET = {
    'mvtec': {
        'repository_root': str(AACLIP_ROOT),
        'image_checkpoint_path': str(TRAIN_ON_VISA_IMAGE),
        'text_checkpoint_path': str(TRAIN_ON_VISA_TEXT) if TRAIN_ON_VISA_TEXT else None,
        'target_dataset': 'mvtec',
    },
    'visa': {
        'repository_root': str(AACLIP_ROOT),
        'image_checkpoint_path': str(TRAIN_ON_MVTEC_IMAGE),
        'text_checkpoint_path': str(TRAIN_ON_MVTEC_TEXT) if TRAIN_ON_MVTEC_TEXT else None,
        'target_dataset': 'visa',
    },
}
print('MVTec:', MVTEC_ROOT)
print('VisA:', VISA_ROOT)
print('MVTec target <- TrainOnVisA:', TRAIN_ON_VISA_IMAGE, TRAIN_ON_VISA_TEXT)
print('VisA target <- TrainOnMVTec:', TRAIN_ON_MVTEC_IMAGE, TRAIN_ON_MVTEC_TEXT)

In [ ]:
from blackbox_evaluation_pipeline import ThresholdCalibrationConfig, calibrate_thresholds

print('===== STEP 3: CALIBRATE ON CLEAN NORMAL TRAINING IMAGES =====')
DATASETS = ('mvtec', 'visa')
THRESHOLD_QUANTILE = 0.95
OUTPUT_ROOT = WORKING / 'aaclip_thresholds_q95'

config = ThresholdCalibrationConfig(
    output_root=str(OUTPUT_ROOT),
    model_name='aaclip',
    model_kwargs_by_target=MODEL_KWARGS_BY_TARGET,
    datasets=DATASETS,
    mvtec_root=str(MVTEC_ROOT),
    visa_root=str(VISA_ROOT),
    device='cuda',
    batch_size=2,
    image_size=518,
    quantile=THRESHOLD_QUANTILE,
    provenance='custom_q95_fallback_official_aaclip_has_no_decision_threshold',
    official_model_threshold=False,
    run_metadata={
        'aaclip_commit': AACLIP_COMMIT,
        'official_repository': AACLIP_REPO_URL,
        'checkpoint_dataset': CHECKPOINT_DATASET_URL,
        'zero_shot_mapping': {
            'mvtec': 'TrainOnVisA',
            'visa': 'TrainOnMVTec',
        },
        'paper_defaults': {
            'model_name': 'ViT-L-14-336',
            'image_size': 518,
            'seed': 111,
            'text_adapt_weight': 0.1,
            'image_adapt_weight': 0.1,
            'text_adapt_until': 3,
            'image_adapt_until': 6,
            'levels': [6, 12, 18, 24],
            'relu': False,
        },
        'calibration_uses_test_data': False,
        'calibration_uses_anomaly_data': False,
    },
)
GENERATED_THRESHOLDS = calibrate_thresholds(config)

In [ ]:
import json

print('===== STEP 4: PRINT GENERATED THRESHOLDS =====')
for dataset, threshold_path in GENERATED_THRESHOLDS.items():
    payload = json.loads(threshold_path.read_text(encoding='utf-8'))
    print(f'\n[{dataset}] {threshold_path}')
    print('category | threshold | count | min | mean | max | std')
    for category, record in payload['categories'].items():
        print(
            f"{category:12s} | {record['threshold']:.6f} | "
            f"{record['sample_count']:5d} | {record['score_min']:.6f} | "
            f"{record['score_mean']:.6f} | {record['score_max']:.6f} | "
            f"{record['score_std']:.6f}"
        )

In [ ]:
print('===== STEP 5: PACKAGE THRESHOLD ARTIFACTS =====')
archive = shutil.make_archive(
    str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name
)
print('Packaged thresholds:', archive)